In [39]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# 하이퍼파라미터 설정
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
DATA_DIR = '../../data/dataset_imagenet'

# 이미지 데이터 생성
# 학습 데이터
train_ds = tf.keras.utils.image_dataset_from_directory(
	DATA_DIR,
	validation_split=0.2,
	subset='training',
	seed=1000,
	image_size=IMG_SIZE,
	batch_size=BATCH_SIZE
)

# 검증 데이터
val_ds = tf.keras.utils.image_dataset_from_directory(
	DATA_DIR,
	validation_split=0.2,
	subset='validation',
	seed=1000,
	image_size=IMG_SIZE,
	batch_size=BATCH_SIZE
)

class_names = train_ds.class_names


Found 14 files belonging to 2 classes.
Using 12 files for training.
Found 14 files belonging to 2 classes.
Using 2 files for validation.


In [40]:
# 성능 최적화
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(20).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [41]:
# 모델 설계
# 전이학습 모델 가져옴
base_model = MobileNetV2(input_shape=(224,224,3),include_top=False, weights='imagenet')

# 데이터 증강=>이미지를돌리고 확대하고 움직여서 여러 이미지를 추가하는 작업 
data_augmentation = tf.keras.Sequential([
	layers.RandomFlip('horizontal_and_vertical'),
	layers.RandomRotation(0.2),
	layers.RandomZoom(0.2)
])

# 새 모델 설계
model = models.Sequential([
	layers.Input((224,224,3)),
	data_augmentation, 
	# 정규화를 0~1이 아닌 -1~1로 해야 함. 왜? imagenet에서 그렇게 했기 때문에 맞춰줘야함
	layers.Rescaling(1./127.5, offset=-1), 
	base_model, 
	layers.GlobalAveragePooling2D(),
	layers.Dense(128, activation='relu'),
	layers.Dropout(0.2),
	layers.Dense(2, activation='softmax')
])

In [42]:
# 모델 설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
	metrics=['accuracy'])

In [43]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 28s 28s/step - accuracy: 0.5000 - loss: 0.7525 - val_accuracy: 1.0000 - val_loss: 0.0700
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0381 - val_accuracy: 1.0000 - val_loss: 0.0130
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0105 - val_accuracy: 1.0000 - val_loss: 0.0029
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0067 - val_accuracy: 1.0000 - val_loss: 7.5206e-04
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 5.5546e-04 - val_accuracy: 1.0000 - val_loss: 2.5242e-04
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0100 - val_accuracy: 1.0000 - val_loss: 7.1401e-05
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 4.8151e-04 - val_accuracy: 1.0000 - val_loss: 2.0802e-05
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0126 - val_accuracy: 1.0000 - val_los

In [44]:
import requests
import numpy as np
from io import BytesIO
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import mobilenet_v2

def predicts_url(url):
	# 이미지 정보를 가져옴
	response = requests.get(url)
	# 가져온 이미지를 바이트 형태로 변환
	img = image.load_img(BytesIO(response.content), target_size=(224,224))

	# 이미지 전처리 => 입력 형태 맞추기, 이미지를 배열로
	X = image.img_to_array(img)
	X = np.expand_dims(X, axis=0)

	# MobileNetV2전용 전처리 
	X = mobilenet_v2.preprocess_input(X)

	# 예측
	predicts = model.predict(X)
	
	print('-'*30)
	for i in range(2):
		predict = predicts[0][i]
		print(f'{class_names[i]} : {predict*100:.2f}%')
	print('-'*30)

In [46]:
# 김밥
predicts_url('https://lh4.googleusercontent.com/proxy/YBicREfizdGpPHIZTWBhzPlJ46-6knDM_k2w7ZeilK4ngmnf1CiG3fc500HGTPIEkw3F9ao4lT-TFheDpiDRwBg')
predicts_url('https://image.greating.co.kr/CO/contents/202505/22/58563F38E9F248DB9AABCC37AB246BBC.jpeg')
# 떡볶이
predicts_url('https://cdn.kihoilbo.co.kr/news/photo/202008/880134_302005_3430.png')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
------------------------------
kimbap : 0.09%
tteokbokki : 99.91%
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
------------------------------
kimbap : 0.07%
tteokbokki : 99.93%
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
------------------------------
kimbap : 0.09%
tteokbokki : 99.91%
------------------------------
